# Comparativa: Limpieza de Datos con Pandas, Polars y Dask

Este notebook demuestra cómo realizar la misma tarea de limpieza de datos del archivo `elecciones_honduras_2017.csv` utilizando tres librerías diferentes. 

**Objetivo:** Cargar el archivo, omitir las primeras 11 filas, usar `;` como separador, eliminar filas nulas y seleccionar un subconjunto de columnas.

### Paso 0: Preparación del Entorno

Primero, descargamos el archivo de datos directamente desde el repositorio de GitHub a nuestro entorno de Colab. Esto asegura que el notebook sea autocontenido y reproducible.

In [ ]:
!wget https://raw.githubusercontent.com/NORSAB/El-Puente-al-Machine-Learning/main/datos/elecciones_honduras_2017.csv

---

### Solución 1: Pandas (La Vía Conocida)

Nuestra herramienta de confianza. Ideal para la mayoría de los casos de uso en una máquina estándar. Su sintaxis es muy intuitiva y es el estándar de la industria.

In [ ]:
import pandas as pd

ruta_elecciones = 'elecciones_honduras_2017.csv'

columnas_deseadas = [
    'DEPARTAMENTO', 
    'MUNICIPIO', 
    'PARTIDO ALIANZA PATRIOTICA HONDUREÑA',
    'PARTIDO LIBERAL DE HONDURAS',
    'PARTIDO NACIONAL DE HONDURAS',
    'BLANCOS', 
    'NULOS', 
    'FECHA DE RECEPCION ',
    'LATITUD', 
    'LONGITUD'
]

# Leemos el CSV con los parámetros necesarios
df_pandas = pd.read_csv(
    ruta_elecciones, 
    skiprows=11, 
    engine='python', 
    sep=';', 
    on_bad_lines='warn'
)
df_pandas.dropna(how='all', inplace=True)

# Seleccionamos las columnas
df_pandas_limpio = df_pandas[columnas_deseadas]

print("Resultado con Pandas:")
display(df_pandas_limpio.head())

---

### Solución 2: Polars (El Retador Veloz)

Una alternativa moderna y extremadamente rápida, ideal para maximizar el rendimiento. Su sintaxis es un poco diferente pero muy poderosa y eficiente.

In [ ]:
# En Colab, podemos instalar librerías directamente con !pip
!pip install polars

import polars as pl

# Polars tiene una sintaxis similar para leer CSVs
df_polars = pl.read_csv(
    ruta_elecciones,
    skip_rows=11,
    separator=';',
    try_parse_dates=True
)

# En Polars, las transformaciones se encadenan de forma muy eficiente.
# Es el equivalente a nuestros dos pasos de Pandas en uno solo.
df_polars_limpio = df_polars.drop_nulls().select(columnas_deseadas)

print("Resultado con Polars:")
display(df_polars_limpio.head())

---

### Solución 3: Dask (Para los Grandes Volúmenes)

La solución para datasets que son demasiado grandes para la memoria RAM. Dask divide los datos en particiones y procesa en paralelo. Su sintaxis es casi idéntica a la de Pandas.

In [ ]:
# Instalamos Dask y sus dependencias para el manejo de dataframes
!pip install "dask[dataframe]"

import dask.dataframe as dd

# La lectura es casi idéntica a Pandas, pero Dask es "perezoso" (lazy)
# No carga los datos hasta que se lo pedimos explícitamente.
df_dask = dd.read_csv(
    ruta_elecciones,
    skiprows=11,
    sep=';',
    on_bad_lines='warn',
    assume_missing=True, # Ayuda a Dask a inferir tipos de datos con nulos
    blocksize=None # Leemos el archivo como una sola partición para este ejemplo
)

# Las transformaciones también son perezosas
df_dask = df_dask.dropna(how='all')
df_dask_limpio = df_dask[columnas_deseadas]

# Le pedimos a Dask que "calcule" el resultado final con .compute()
resultado_dask = df_dask_limpio.compute()

print("Resultado con Dask:")
display(resultado_dask.head())